# Hyperparameter Tuning using HyperDrive

TODO: Import Dependencies. In the cell below, import all the dependencies that you will need to complete the project.

In [1]:
from azureml.core import Workspace, Experiment, Environment, ScriptRunConfig
from azureml.train.sklearn import SKLearn
from azureml.train.hyperdrive.runconfig import HyperDriveConfig, PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.parameter_expressions import choice
from azureml.widgets import RunDetails

## Dataset

TODO: Get data. In the cell below, write code to access the data you will be using in this project. Remember that the dataset needs to be external.

In [2]:
ws = Workspace.from_config()
experiment_name = 'capstone-project'

experiment=Experiment(ws, experiment_name)

In [6]:
# create a compute cluster
from azureml.core.compute import ComputeTarget, AmlCompute

cluster_name = "compute-cluster"
configuration = AmlCompute.provisioning_configuration(vm_size = "Standard_D4a_v4",
                                                      max_nodes=1,
                                                      vm_priority='dedicated',
                                                      idle_seconds_before_scaledown=600) # use for private azure account, west germany
cluster = ComputeTarget.create(workspace=ws,name=cluster_name,provisioning_configuration=configuration)

## Hyperdrive Configuration

TODO: Explain the model you are using and the reason for chosing the different hyperparameters, termination policy and config settings.

In [25]:
# TODO: Create an early termination policy. This is not required if you are using Bayesian sampling.
early_termination_policy = BanditPolicy(evaluation_interval=2, slack_factor = 0.1)

#TODO: Create the different params that you will be using during training
param_sampling = RandomParameterSampling(parameter_space ={"--n_estimators": choice(50, 100, 200),
                                          "--max_depth": choice(-1, 5, 10), # -1 as None value
                                          "--min_samples_split": choice(2, 5, 10),
                                          "--min_samples_leaf": choice(1, 2, 4),
                                          "--max_features": choice("auto", 'sqrt', 'log2')}) # "" as None value

#TODO: Create your estimator and hyperdrive config
sklearn_env = Environment.from_conda_specification(name='sklearn-env', file_path='conda_dependencies.yml')

src = ScriptRunConfig(source_directory="./", script="train.py",compute_target=cluster, environment=sklearn_env,)

hyperdrive_config = HyperDriveConfig(hyperparameter_sampling = param_sampling,
                                         policy=early_termination_policy,
                                         run_config=src,
                                         primary_metric_name="Accuracy",
                                         primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
                                         max_total_runs=150,
                                         max_duration_minutes = 60)

In [26]:
#TODO: Submit your experiment
hyperdrive_run = experiment.submit(config=hyperdrive_config)

## Run Details

OPTIONAL: Write about the different models trained and their performance. Why do you think some models did better than others?

TODO: In the cell below, use the `RunDetails` widget to show the different experiments.

In [34]:
print(hyperdrive_run.get_portal_url())
RunDetails(hyperdrive_run).show()

https://ml.azure.com/runs/HD_31ba53a9-7eba-4a3e-bea8-c1e61d97c01f?wsid=/subscriptions/d93ce4ff-fc1a-4ad9-a9c7-1df3382f954d/resourcegroups/resource-group/workspaces/azure-ml&tid=796de0a1-82b9-44ae-9691-75802bf973fa


_HyperDriveWidget(widget_settings={'childWidgetDisplay': 'popup', 'send_telemetry': False, 'log_level': 'INFO'…

## Best Model

TODO: In the cell below, get the best model from the hyperdrive experiments and display all the properties of the model.

In [38]:
hyperdrive_run.wait_for_completion()
best_run = hyperdrive_run.get_best_run_by_primary_metric()
best_run_metrics = best_run.get_metrics()
parameter_values = best_run.get_details()['runDefinition']['arguments']

print('Best Run Id: ', best_run.id)
print('\n Accuracy:', best_run_metrics['Accuracy'])

# print the model parameter values for the best run
print('\n Model parameters: ')
for elem in parameter_values:
    print(elem)

Best Run Id:  HD_31ba53a9-7eba-4a3e-bea8-c1e61d97c01f_36

 Accuracy: 0.9466666666666667

 Model parameters: 
--max_depth
5
--max_features
auto
--min_samples_leaf
2
--min_samples_split
2
--n_estimators
100


In [39]:
#TODO: Save the best model
best_run.register_model(model_name='heart_failure_prediction_best_hyperdrive_model', model_path='outputs/model.pkl')

Model(workspace=Workspace.create(name='azure-ml', subscription_id='d93ce4ff-fc1a-4ad9-a9c7-1df3382f954d', resource_group='resource-group'), name=heart_failure_prediction_best_hyperdrive_model, id=heart_failure_prediction_best_hyperdrive_model:2, version=2, tags={}, properties={})

## Model Deployment

Remember you have to deploy only one of the two models you trained but you still need to register both the models. Perform the steps in the rest of this notebook only if you wish to deploy this model.

TODO: In the cell below, register the model, create an inference config and deploy the model as a web service.

In [37]:
# registration happend in the cell above to save to model

#create inference config and deploy the model as a web service
from azureml.core import Model
from azureml.core.webservice import AciWebservice
from azureml.core.model import InferenceConfig
from azureml.core.environment import Environment

model = Model(ws, 'heart_failure_prediction_best_hyperdrive_model')
env = Environment.from_conda_specification(name='prediction-env', file_path='conda_dependencies.yml')

inference_config = InferenceConfig(entry_script='score.py', environment=env)

service_name = 'heart-failure-prediction-service'
aci_config = AciWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)

service = Model.deploy(workspace = ws,
                       name = service_name,
                       models = [model],
                       inference_config = inference_config,
                       deployment_config = aci_config)

print(f"Service state: {service.state}")
print(f"Scoring URI: {service.scoring_uri}")


entry_script score.py doesn't exist. entry_script should be path relative to current working directory



WebserviceException: WebserviceException:
	Message: entry_script score.py doesn't exist. entry_script should be path relative to current working directory
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "entry_script score.py doesn't exist. entry_script should be path relative to current working directory"
    }
}

TODO: In the cell below, send a request to the web service you deployed to test it.

TODO: In the cell below, print the logs of the web service and delete the service

**Submission Checklist**
- I have registered the model.
- I have deployed the model with the best accuracy as a webservice.
- I have tested the webservice by sending a request to the model endpoint.
- I have deleted the webservice and shutdown all the computes that I have used.
- I have taken a screenshot showing the model endpoint as active.
- The project includes a file containing the environment details.

